# P05 · Momentum y Mean Reversion
**QuantLab | Módulo 1 · Fundamentos financieros | Práctica 05/12**

### Objetivo
Investigar si la evolución pasada de BTC/EUR contiene información sobre sus retornos futuros. Construiremos dos señales, las compararemos mediante correlación y simularemos operaciones long/cash con costes. No se enviarán órdenes.

**Ruta:** datos → indicadores → objetivo futuro → separación temporal → diagnóstico predictivo → simulación → ejercicios. Ejecuta de arriba abajo. Los ejercicios están resueltos en la sección 10. Ningún resultado sintético constituye evidencia sobre Bitcoin.

Las funciones están en **`src/quantlab/p05/signals.py`** y sus pruebas en **`tests/test_signals.py`** (`pytest` desde la raíz del proyecto).

## 1. Configuración
Colab suele incluir estas bibliotecas. Si faltan: `%pip install numpy pandas matplotlib requests`. En Jupyter o VS Code selecciona tu kernel QuantLab. Los parámetros son educativos y se fijan antes de mirar la validación.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display
import inspect
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p05.signals import (KRAKEN_OHLC_URL, SYNTHETIC_REGIME_CHANGE, add_target, breakeven_costs,
                                  build_features, cost_sensitivity, development_grid, fetch_kraken_ohlc,
                                  information_table, load_p04_series, manual_momentum, manual_zscore,
                                  regularize, simulate, split_train_test, strategy_table, synthetic_prices)

DATA_MODE = 'synthetic'  # 'synthetic', 'live', 'csv'
CSV_PATH = 'quantlab_exports/p04_synthetic_REEMPLAZAR_series.csv'
WINDOW = '1min'  # debe coincidir con P04 si importas CSV
PAIR = 'XBTEUR'
INTERVAL_MIN = 1  # solo para API Kraken; debe coincidir con WINDOW
LOOKBACK = 20
HORIZON = 5
Z_THRESHOLD = 1.5
TRAIN_FRACTION = .65
ROUND_TRIP_BPS = 20.0  # supuesto, no tarifa real
EXPORT = False
URL = KRAKEN_OHLC_URL  # 'https://api.kraken.com/0/public/OHLC'
assert LOOKBACK >= 2 and HORIZON >= 1 and 0 < TRAIN_FRACTION < 1
assert ROUND_TRIP_BPS >= 0 and Z_THRESHOLD > 0
plt.rcParams.update({'figure.figsize': (10, 3.4), 'axes.grid': True,
                     'grid.alpha': .2, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})

## 2. Datos y criterios de calidad
- **`synthetic`:** precios ficticios reproducibles, con cambios de dependencia temporal introducidos deliberadamente y un hueco. No es un modelo calibrado de BTC.
- **`live`:** velas recientes BTC/EUR del endpoint público OHLC de Kraken, sin claves. La API devuelve hasta 720 velas; descartamos la última, aún abierta. Es una muestra pequeña, no un histórico suficiente para demostrar una ventaja.
- **`csv`:** selecciona el `*_series.csv` exportado por P04. Conservamos `analysis_close`, que excluye las fronteras y ventanas vacías. Activa `EXPORT=True` en P04 para generarlo; sube el archivo a Colab si procede y ajusta `CSV_PATH` y `WINDOW`.

El índice es el fin de la vela en UTC, cuando su información queda disponible como pronto. No rellenamos huecos ni mezclamos distintas frecuencias. Para el CSV, comprueba el fichero de metadatos de P04 y que corresponda a un único mercado.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿la serie de precios es coherente?</b>

`regularize` se detiene con un error si:

- El índice está vacío, tiene fechas duplicadas o vacías, o la ventana no es positiva.
- Algún timestamp no encaja en la rejilla de `WINDOW` (por ejemplo, 10:00:30 con velas de 1 minuto).
- Hay precios ≤ 0 o no finitos.

Después pasa la serie a frecuencia fija: los intervalos que faltan aparecen como `NaN`. **No se rellenan.**

</div>

In [ ]:
print(inspect.getsource(regularize))

loaders = {'synthetic': lambda: synthetic_prices(WINDOW),
           'live': lambda: fetch_kraken_ohlc(PAIR, INTERVAL_MIN, WINDOW),  # si falla, lanza el error
           'csv': lambda: load_p04_series(CSV_PATH)}
close, provenance = loaders[DATA_MODE]()
close = regularize(close, WINDOW)
print(provenance)
print(f'{len(close)} ventanas | {close.isna().sum()} huecos | {close.index.min()} — {close.index.max()}')
display(close.head().to_frame())

## 3. Construcción de indicadores
**Momentum:** $M_t=P_t/P_{t-N}-1$. Si es positivo, el precio ha subido en ese horizonte; no predice por sí mismo el futuro.

**Media simple:** $SMA_t=\frac{1}{N}\sum_{j=0}^{N-1}P_{t-j}$. **Desviación:** $D_t=P_t-SMA_t$. **Z-score:** $z_t=D_t/s_t$, con desviación típica muestral de los precios de esa ventana. Un z-score bajo indica que el precio está bajo su media, no que esté infravalorado económicamente. Un precio no estacionario no tiene por qué revertir.

La **EMA** pondera más los precios recientes. Reiniciamos su cálculo tras cada hueco. Los indicadores solo utilizan datos hasta el cierre actual.

In [ ]:
print(inspect.getsource(build_features))

features = build_features(close, LOOKBACK)
display(features.dropna().head().round(5))
fig, ax = plt.subplots()
tail = features.tail(240)
ax.plot(tail.index,tail.close,label='Cierre',color='#264b69')
ax.plot(tail.index,tail.sma,label='SMA',color='#b36a37')
ax.plot(tail.index,tail.ema,label='EMA',color='#528c88',linestyle='--')
ax.set(title=f'{DATA_MODE} · Precio y medias (N={LOOKBACK})',ylabel='EUR/BTC',xlabel='UTC'); ax.legend()
fig.autofmt_xdate(); fig.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(features.index, features.z_score, color='#32658a',linewidth=.9)
for level in [-Z_THRESHOLD,0,Z_THRESHOLD]: ax.axhline(level,color='#b36a37' if level else 'gray',linestyle='--',linewidth=.9)
ax.set(title=f'{DATA_MODE} · Desviación estandarizada del precio',ylabel='Z-score (sin unidad)',xlabel='UTC')
fig.autofmt_xdate(); fig.tight_layout(); plt.show()

## 4. Objetivo futuro y alineación
La señal se observa al cierre $t$. Para no asumir ejecución instantánea usamos entrada al cierre de $t+1$ y salida al cierre de $t+1+h$. El retorno objetivo es $y_t=P_{t+1+h}/P_{t+1}-1$. Son precios de referencia, no ejecuciones garantizadas. Exigimos que todas las ventanas entre señal y salida tengan precio observado.

Ejemplo con h=2: señal 10:01, entrada de referencia 10:02, salida 10:04. No se puede usar `forward_return`, `entry_price` o `exit_price` como variables predictoras.

In [ ]:
print(inspect.getsource(add_target))

dataset = add_target(features,HORIZON)
display(dataset[['close','momentum','z_score','entry_time','exit_time','forward_return']].dropna().head(3))

## 5. Comprobaciones matemáticas y temporales
Usamos datos de resultado conocido y verificamos que truncar la serie no cambia las características históricas. Los huecos no se convierten en observaciones ficticias.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: fórmulas, huecos, causalidad de las features y fechas del objetivo
probe = pd.Series([100.,102.,104.,106.,108.,110.,np.nan,112.,114.],index=pd.date_range('2026-01-01',periods=9,freq='1min',tz='UTC'))
f = build_features(probe,2)
t = add_target(f,2)
assert np.isclose(f.momentum.iloc[2],.04)
assert np.isclose(f.sma.iloc[2],103)
assert np.isclose(f.rolling_std.iloc[2],np.sqrt(2))
assert np.isclose(f.z_score.iloc[2],1/np.sqrt(2))
assert np.isclose(t.forward_return.iloc[0],106/102-1)
assert pd.isna(t.forward_return.iloc[3])
assert pd.isna(f.momentum.iloc[7])
assert pd.isna(build_features(pd.Series([1.,1.,1.]),2).z_score.iloc[-1])
cut = len(close)//2
pd.testing.assert_frame_equal(build_features(close.iloc[:cut],LOOKBACK),features.iloc[:cut],check_freq=False)
valid = dataset.dropna(subset=['forward_return'])
assert (valid.entry_time > valid.index).all() and (valid.exit_time > valid.entry_time).all()
print('10 comprobaciones superadas: fórmulas, huecos, causalidad de features y fechas.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Las features solo miran al pasado</b>

- **Fórmulas** (precios 100, 102, 104 con N = 2): momentum = 104/100 − 1 = **4 %**, SMA = **103**, desviación típica = **√2**, z = 1/√2 ≈ **0,71**.
- **Huecos:** el objetivo de la fila 3 (106) no existe, porque su salida caería después del hueco. El momentum de la fila 7 tampoco: necesitaría el precio del hueco.
- **Precio constante:** la desviación típica es 0 y el z-score queda **indefinido**, no infinito.
- **Causalidad:** calcular las features con la mitad de la serie da exactamente lo mismo que con la serie completa. Si cambiaran, estarían usando datos futuros.
- **Fechas:** la entrada es siempre posterior a la señal, y la salida, posterior a la entrada.

📄 Ver `docs/conceptos/momentum_y_reversion.md`

</div>

## 6. Separar descubrimiento y validación
Usamos el primer 65 % cronológico como desarrollo y el resto como validación. Una fila de desarrollo se elimina si su salida toca o cruza la frontera. Las características de validación pueden usar el pasado anterior a la frontera, pero nunca resultados futuros. Parámetros fijos: si eliges otros mirando la validación, necesitarás un nuevo periodo de test independiente. Las etiquetas de distintos instantes pueden solaparse: no interpretaremos sus correlaciones como pruebas estadísticas independientes.

In [ ]:
print(inspect.getsource(split_train_test))

train, test, split_time = split_train_test(dataset, TRAIN_FRACTION)
# 🔎 FASE DE VALIDACIÓN: ninguna salida del desarrollo llega a la validación
assert train.empty or train.exit_time.max() < split_time
assert test.empty or test.index.min() >= split_time
print('Frontera UTC:',split_time,'| Desarrollo:',len(train),'| Validación:',len(test))
if min(len(train),len(test)) < 30:
    print('AVISO: muestra insuficiente; usa un CSV más largo. No interpretes la escasez como ausencia de ventaja.')

## 7. Capacidad predictiva
El IC (Pearson) mide asociación lineal entre señal y retorno futuro. También calculamos Spearman para estudiar asociación por rangos. Ambos son diagnósticos descriptivos; no calculamos p-valores ingenuos porque los retornos futuros se solapan. La señal de reversión es −z_score, por tanto ambas señales esperan retornos positivos cuando su valor aumenta.

Los grupos extremos se definen con cuantiles del desarrollo, nunca con los de validación. Comparamos el retorno medio de ambos extremos y la muestra global.

In [ ]:
print(inspect.getsource(information_table))

signals = ['momentum_signal','reversion_signal']
ic_table = information_table(train, test, signals)
display(ic_table.round(4))
fig, ax = plt.subplots()
plot = ic_table.pivot(index='signal',columns='period',values='ic_pearson')
plot.plot.bar(ax=ax,rot=0,color=['#32658a','#b36a37'])
ax.axhline(0,color='gray',linewidth=.8); ax.set(title=f'{DATA_MODE} · IC por periodo',ylabel='Correlación de Pearson',xlabel='Señal')
fig.tight_layout(); plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · IC en la simulación (N = 20, h = 5)</b>

| Señal | Periodo | n | IC Pearson | IC Spearman | Media global | Extremo bajo | Extremo alto | Alto − bajo |
|---|---|---|---|---|---|---|---|---|
| Momentum | Desarrollo | 1.115 | −0,089 | −0,082 | −2,04 bps | −0,94 bps | −7,51 bps | **−6,57 bps** |
| Momentum | Validación | 624 | −0,045 | −0,016 | −2,38 bps | +1,01 bps | −3,18 bps | −4,19 bps |
| Reversión | Desarrollo | 1.115 | +0,026 | +0,035 | −2,04 bps | −2,03 bps | +0,02 bps | +2,04 bps |
| Reversión | Validación | 624 | **−0,018** | −0,015 | −2,38 bps | −1,15 bps | −1,90 bps | −0,75 bps |

- **Momentum** tiene IC negativo en los dos periodos: con estos parámetros se comporta más bien como señal contraria.
- **Reversión** cambia de signo: +0,026 en desarrollo y −0,018 en validación. **No persiste.**
- Todos los IC son muy pequeños (|IC| < 0,09).
- En validación, los grupos extremos ya no tienen el 20 % de las filas (46 y 44 en momentum), porque sus cuantiles se fijaron en el desarrollo. Es lo correcto: recalcularlos con la validación sería mirar el resultado.

</div>

## 8. De la señal a una estrategia long/cash
Reglas educativas, no recomendaciones: **Momentum** entra si momentum > 0; **Mean Reversion** entra si z_score < −Z_THRESHOLD. Entrada y salida con el retraso y horizonte definidos. No abrimos posiciones simultáneas dentro de cada estrategia: la siguiente entrada debe ser posterior a la salida anterior. Capital sin apalancamiento; reinversión completa ficticia; efectivo sin intereses. El coste total por ciclo es un supuesto fijo en bps sobre nominal de entrada: `net_return = gross_return - cost_bps/10000` (aproximación).

La referencia `periodic_long` aplica la misma duración y costes sin señal, siempre que haya una fila elegible. Las estrategias tienen distinta frecuencia y exposición; no equivale a Alpha ajustado por riesgo. No modelamos spread dinámico, llenados, fiscalidad ni impacto. Compararemos con costes de 0, 10, 20 y 40 bps sin ajustar la regla.

In [ ]:
print(inspect.getsource(simulate))

rules=['momentum','mean_reversion','periodic_long']
periods = {'desarrollo': train, 'validación': test}
strategy_results, trade_logs = strategy_table(periods, ROUND_TRIP_BPS, Z_THRESHOLD, rules)
display(strategy_results.round(3))
# 🔎 FASE DE VALIDACIÓN: cada operación paga exactamente el coste supuesto
assert all(log.empty or np.allclose(log.gross_return-log.net_return,ROUND_TRIP_BPS/10000) for log in trade_logs.values())

In [ ]:
fig, ax=plt.subplots()
for rule,color in zip(rules,['#32658a','#b36a37','#528c88']):
    tr=trade_logs[('validación',rule)]
    x=[split_time]+tr.exit_time.tolist()
    y=[1000]+(1000*tr.equity).tolist()
    ax.step(x,y,where='post',label=rule,color=color)
ax.set(title=f'{DATA_MODE} · Validación: capital tras cierres de operaciones, coste {ROUND_TRIP_BPS:g} bps',xlabel='UTC',ylabel='Capital simulado (€)'); ax.legend()
fig.autofmt_xdate(); fig.tight_layout(); plt.show()
# No es valoración mark-to-market durante posiciones abiertas; no usar para medir drawdown intraposición.

In [ ]:
cost_table = cost_sensitivity(test, [0,10,20,40], Z_THRESHOLD, rules)
display(cost_table.pivot(index='rule',columns='cost_bps',values='mean_net_bps').round(3))
fig,ax=plt.subplots()
for rule,group in cost_table.groupby('rule'):
    ax.plot(group.cost_bps,group.mean_net_bps,marker='o',label=rule)
ax.axhline(0,color='gray',linewidth=.8); ax.set(title=f'{DATA_MODE} · Sensibilidad a costes en validación',xlabel='Coste total (bps)',ylabel='Retorno neto medio por operación (bps)'); ax.legend()
fig.tight_layout(); plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Simulación long/cash con 20 bps por ciclo</b>

| Periodo | Regla | Operaciones | Bruto medio | Neto medio | % ganadoras | Total neto |
|---|---|---|---|---|---|---|
| Desarrollo | Momentum | 95 | −0,33 bps | −20,33 bps | 21,1 % | **−17,6 %** |
| Desarrollo | Mean reversion | 56 | −0,04 bps | −20,04 bps | 21,4 % | −10,6 % |
| Desarrollo | Periodic long | 187 | −1,24 bps | −21,24 bps | 23,0 % | −32,9 % |
| Validación | Momentum | 51 | −1,66 bps | −21,66 bps | 11,8 % | −10,5 % |
| Validación | Mean reversion | 36 | −5,47 bps | −25,47 bps | 5,6 % | −8,8 % |
| Validación | Periodic long | 104 | −3,39 bps | −23,39 bps | 13,5 % | −21,6 % |

- **Todas pierden, incluso antes de costes:** el retorno bruto medio es negativo en los seis casos. Con 20 bps por ciclo, la pérdida se dispara.
- **Los costes dominan:** un movimiento bruto de unos pocos bps no cubre nunca 20 bps. Con coste 0, el neto en validación sigue siendo negativo (−1,66, −5,47 y −3,39 bps).
- **Más operaciones = más costes:** `periodic_long` pierde más en total (−21,6 %) porque opera el doble que las otras.

</div>

## 9. Interpretación guiada
Responde observando las tablas, no por intuición. ¿Mantiene el IC su signo en validación? ¿Supera el retorno del grupo alto al bajo? ¿Cuántas operaciones tenemos y cuánto se pierde por costes? El cambio artificial de dependencia en la simulación puede modificar los resultados entre periodos. Un IC positivo no implica un beneficio neto positivo. Un único periodo de validación tampoco demuestra robustez. No hay contraste de significancia ni estimación de intervalos ajustada por dependencia; los incorporaremos en módulos posteriores.

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Interpretación · La dependencia existe, pero el retraso de entrada la elimina</b>

**¿Mantiene el IC su signo en validación?** Momentum sí (negativo en los dos periodos); reversión no (+0,026 → −0,018).

**¿Supera el grupo alto al bajo?** Solo en la reversión del desarrollo (+2,04 bps), y se invierte en validación (−0,75 bps). En momentum, el alto rinde menos que el bajo en los dos periodos.

**¿Operaciones y costes?** Entre 36 y 187 operaciones por caso. Los 20 bps de coste son entre 4 y casi 600 veces el retorno bruto medio (en valor absoluto).

**¿Por qué la simulación, que tiene dependencia temporal, no da señal?** La simulación es un AR(1) de retornos: φ = +0,35 en las primeras 900 velas y φ = −0,35 después (el cambio cae dentro del desarrollo, a las 01:01 del 2 de enero). La dependencia es real:

| | Autocorrelación de retardo 1 | IC (Spearman) de "último retorno → retorno siguiente" sin retraso |
|---|---|---|
| Régimen 1 (φ = +0,35) | **+0,40** | **+0,37** |
| Régimen 2 (φ = −0,35) | **−0,36** | **−0,35** |

Pero el objetivo empieza en **t + 1**: la entrada se retrasa una vela para no suponer una ejecución instantánea. La dependencia de un solo retardo afecta justamente al retorno de t a t + 1, que queda **fuera** del objetivo. A partir de ahí la información decae como φ² ≈ 0,12. Además, la reversión sí tuvo IC +0,25 en el tramo del desarrollo posterior al cambio de régimen, pero ese efecto no se trasladó a la validación con estos parámetros.

**Lección:** una dependencia estadística real no implica una señal **explotable**. El retraso de ejecución, el horizonte y los costes pueden hacerla desaparecer.

</div>

## 10. Ejercicios
**E1 — Momentum manual.** Implementa una función sobre [100, 102, 104, 106] con N=2; comprueba la primera observación disponible (4 %). Explica las dos primeras ausencias.

**E2 — Z-score.** Calcula SMA, desviación muestral y z-score de [100, 102, 104]. Interpreta un z-score negativo. ¿Por qué no es un diagnóstico de infravaloración?

**E3 — Alineación.** Verifica manualmente tres filas (fecha señal, entrada, salida). Construye un hueco y comprueba que un objetivo no lo atraviesa.

**E4 — Predicción.** En desarrollo compara N=5,20,60 y h=1,5,10 usando IC y número de observaciones. Elige una combinación sin consultar validación; congela parámetros y evalúa una única vez en un nuevo periodo si ya has visto esta validación. No selecciones por el mejor resultado de test.

**E5 — Economía.** Calcula el coste de equilibrio por operación a partir del retorno bruto medio. Si es negativo, explica por qué ni coste cero basta. Compara con `periodic_long` y señala sus distintas exposiciones.

**E6 — Datos reales.** Usa `DATA_MODE='live'` o un CSV largo de P04. Documenta mercado, fechas, frecuencia, huecos, IC y retorno neto por periodo. Si no hay suficientes datos, tu conclusión debe ser 'evidencia insuficiente'. No ajustes hasta conseguir beneficios.

**E7 — Investigación.** Redacta una hipótesis falsable para combinar Momentum y Microprice de P02. Define qué datos simultáneos te faltarían, horizonte y criterio de rechazo; no unas snapshots aislados por su número de fila.

### E1 · Momentum manual

In [ ]:
print(inspect.getsource(manual_momentum))

e1 = manual_momentum([100, 102, 104, 106], 2)
display(pd.DataFrame({'precio': [100, 102, 104, 106], 'momentum N=2': e1}))
# 🔎 FASE DE VALIDACIÓN: coincide con el cálculo a mano y con build_features
assert e1[:2] == [None, None] and np.isclose(e1[2], .04)
assert np.allclose(build_features(pd.Series([100., 102., 104., 106.]), 2).momentum.iloc[2:], e1[2:])
print('E1 verificado: primera observación = 104/100 − 1 = 4 %.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · Las primeras N filas no tienen pasado suficiente</b>

| t | Precio | Momentum (N = 2) | Por qué |
|---|---|---|---|
| 0 | 100 | — | No existe P_{t−2} |
| 1 | 102 | — | No existe P_{t−2} |
| 2 | 104 | 104 / 100 − 1 = **+4,00 %** | Primera observación con 2 precios hacia atrás |
| 3 | 106 | 106 / 102 − 1 = **+3,92 %** | |

- Las dos primeras ausencias **no son ceros**: no hay datos para calcularlas. Rellenarlas con 0 diría "el precio no se movió", que es falso.
- En `build_features` se exigen además N + 1 cierres sin huecos en la ventana. Si uno falta, el momentum queda indefinido.

</div>

### E2 · SMA, desviación típica y z-score

In [ ]:
print(inspect.getsource(manual_zscore))

e2 = manual_zscore([100, 102, 104])
display(pd.Series(e2).to_frame('[100, 102, 104]'))
e2_low = manual_zscore([104, 102, 100])
display(pd.Series(e2_low).to_frame('[104, 102, 100]'))
assert np.isclose(e2['sma'], 102) and np.isclose(e2['std'], 2) and np.isclose(e2['z_score'], 1)
print('E2 verificado: SMA = 102, s = 2, z = (104 − 102) / 2 = 1.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · Un z-score bajo es una posición estadística, no un precio barato</b>

- **SMA** = (100 + 102 + 104) / 3 = **102**.
- **Desviación típica muestral** = √(((−2)² + 0² + 2²) / (3 − 1)) = √4 = **2**. Se divide entre n − 1 porque es una estimación a partir de una muestra.
- **z-score** del último precio = (104 − 102) / 2 = **+1**: está una desviación típica por **encima** de su media. Con la serie inversa [104, 102, 100], z = **−1**.
- **z negativo** significa que el precio actual está por debajo de su media reciente, medido en desviaciones típicas.
- **No es infravaloración** porque no compara el precio con ningún valor fundamental: solo con su propio pasado reciente. Si el precio tiene tendencia (no es estacionario), puede seguir cayendo y no volver nunca a esa media. La "reversión a la media" es una **hipótesis** que hay que probar, no una propiedad garantizada.

</div>

### E3 · Alineación temporal y huecos

In [ ]:
rows = dataset.dropna(subset=['forward_return']).iloc[[0, 1, 2]]
manual = rows[['close', 'entry_time', 'entry_price', 'exit_time', 'exit_price', 'forward_return']].copy()
manual['check'] = [close.loc[r.exit_time] / close.loc[r.entry_time] - 1 for r in rows.itertuples()]
display(manual)
assert np.allclose(manual.check, manual.forward_return)
assert ((manual.entry_time - manual.index) == pd.Timedelta(WINDOW)).all()
assert ((manual.exit_time - manual.entry_time) == HORIZON * pd.Timedelta(WINDOW)).all()

# Hueco construido a mano: con h = 2, ningún objetivo debe atravesarlo
gap_probe = pd.Series([100., 101., 102., 103., np.nan, 105., 106., 107., 108.],
                      index=pd.date_range('2026-01-01 10:01', periods=9, freq='1min', tz='UTC'))
gap_target = add_target(build_features(gap_probe, 2), 2)
display(gap_target[['close', 'entry_time', 'exit_time', 'forward_return']])
gap_time = gap_probe.index[gap_probe.isna()][0]
crossing = gap_target.dropna(subset=['forward_return'])
assert not ((crossing.index <= gap_time) & (crossing.exit_time >= gap_time)).any()
print('E3 verificado: entrada = señal + 1 vela, salida = entrada + h velas, y ningún objetivo cruza el hueco.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · Señal, entrada y salida siempre en este orden</b>

Tres filas de la simulación (h = 5):

| Señal | Entrada | Salida | Precio de entrada | Precio de salida | Objetivo |
|---|---|---|---|---|---|
| 10:01 | 10:02 | 10:07 | 59.943,87 € | 59.809,88 € | −0,224 % |
| 10:02 | 10:03 | 10:08 | 59.964,72 € | 59.783,06 € | −0,303 % |
| 10:03 | 10:04 | 10:09 | 60.022,81 € | 59.772,76 € | −0,417 % |

- La entrada es **una vela después** de la señal, y la salida, **h velas** después de la entrada. El objetivo recalculado a mano coincide.
- **Hueco construido** (precio de las 10:05 ausente, h = 2): la señal de las 10:01 sí tiene objetivo (sale a las 10:04, antes del hueco), pero las de las 10:02 a las 10:05 no, porque su ventana de señal a salida tocaría el hueco. La primera con objetivo después del hueco es la de las 10:06.
- En la serie sintética, el hueco de 17:31 a 17:33 deja sin objetivo las señales de 17:25 a 17:30.

</div>

### E4 · Rejilla de parámetros, solo en desarrollo

In [ ]:
print(inspect.getsource(development_grid))

e4 = development_grid(close, (5, 20, 60), (1, 5, 10), TRAIN_FRACTION)
display(e4.round(4))
display(e4.pivot(index='lookback', columns='horizon', values='ic_reversion').round(4).rename_axis('IC reversión'))

In [ ]:
# Regla fijada ANTES de mirar resultados: para cada señal, el mayor IC de desarrollo con al menos 1.000 filas.
eligible_grid = e4[e4.n_train >= 1000]
frozen = {sig: eligible_grid.loc[eligible_grid[f'ic_{sig}'].idxmax(), ['lookback', 'horizon']].astype(int).to_dict()
          for sig in ['momentum', 'reversion']}
print('Parámetros congelados:', frozen)

# Evaluación única. AVISO: esta validación ya se vio en la sección 7, así que NO es un periodo independiente.
frozen_rows = []
for sig, params in frozen.items():
    _, frozen_test, _ = split_train_test(add_target(build_features(close, params['lookback']), params['horizon']), TRAIN_FRACTION)
    frozen_rows.append({'signal': sig, **params, 'n_test': len(frozen_test),
                        'ic_validation': frozen_test[f'{sig}_signal'].corr(frozen_test.forward_return)})
display(pd.DataFrame(frozen_rows).round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · Elegir en desarrollo y congelar</b>

IC de desarrollo en la simulación:

| N \\ h | Momentum h = 1 | h = 5 | h = 10 | Reversión h = 1 | h = 5 | h = 10 |
|---|---|---|---|---|---|---|
| 5 | **+0,026** | −0,006 | −0,020 | −0,067 | +0,035 | +0,030 |
| 20 | −0,041 | −0,089 | −0,113 | −0,029 | +0,026 | +0,064 |
| 60 | −0,034 | −0,084 | −0,129 | +0,003 | +0,073 | **+0,114** |

Observaciones de desarrollo: de 1.025 (N = 60, h = 10) a 1.153 (N = 5, h = 1). Con N y h mayores se pierden más filas al principio, al final y alrededor del hueco.

- **Regla de elección, fijada antes de mirar:** mayor IC de desarrollo con al menos 1.000 observaciones.
- **Congelados:** momentum con **N = 5, h = 1** y reversión con **N = 60, h = 10**.
- **Evaluación única:** momentum +0,060 y reversión **−0,004** en validación. La reversión, la "mejor" del desarrollo, desaparece.
- ⚠️ **Esta validación ya se había visto** en la sección 7: no es un periodo independiente. La evaluación honesta de estos parámetros congelados exige datos nuevos. En E6 se aplican a 12 horas reales de Kraken: momentum +0,054 y reversión +0,162, sobre solo 250 y 241 filas solapadas. Son muy pocas para concluir nada.
- Con 9 combinaciones probadas, alguna tendrá un IC "alto" solo por azar. Es la razón para no elegir mirando el test.

</div>

### E5 · Coste de equilibrio y exposición

In [ ]:
print(inspect.getsource(breakeven_costs))

e5 = breakeven_costs(trade_logs, periods)
display(e5.round(3))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · Si el bruto es negativo, ni coste cero basta</b>

Con la aproximación `net = gross − coste`, el **coste de equilibrio** es el retorno bruto medio por operación:

| Periodo | Regla | Operaciones | Coste de equilibrio | Exposición |
|---|---|---|---|---|
| Desarrollo | Momentum | 95 | **−0,33 bps** | 41,6 % |
| Desarrollo | Mean reversion | 56 | −0,04 bps | 24,5 % |
| Desarrollo | Periodic long | 187 | −1,24 bps | **81,8 %** |
| Validación | Momentum | 51 | −1,66 bps | 40,9 % |
| Validación | Mean reversion | 36 | −5,47 bps | 28,9 % |
| Validación | Periodic long | 104 | −3,39 bps | 83,5 % |

- Los seis costes de equilibrio son **negativos**: habría que **cobrar** por operar para no perder. Con coste 0 ya se pierde, porque las operaciones elegidas tienen de media un retorno bruto negativo.
- **Exposiciones distintas:** `periodic_long` está invertido más del 80 % del tiempo, momentum un 41 % y mean reversion un 25–29 %. Comparar sus resultados totales mezcla la calidad de la señal con el tiempo expuesto al mercado.
- En la simulación el precio baja en conjunto (media global ≈ −2 bps por operación), así que estar más tiempo invertido perjudica a `periodic_long`. Eso es efecto del mercado, no de la ausencia de señal.

</div>

### E6 · Datos reales de Kraken
Cambia `LIVE_E6` a `True` para descargar las últimas 720 velas de 1 minuto y repetir el análisis con los mismos parámetros. Si la conexión falla, la celda se detiene con el error.

In [ ]:
LIVE_E6 = False

if LIVE_E6:
    live_close, live_prov = fetch_kraken_ohlc(PAIR, INTERVAL_MIN, WINDOW)  # si falla, lanza el error
    live_close = regularize(live_close, WINDOW)
    print(live_prov)
    print(f'{len(live_close)} velas de {WINDOW} | {int(live_close.isna().sum())} huecos | '
          f'{live_close.index.min()} — {live_close.index.max()}')
    live_ds = add_target(build_features(live_close, LOOKBACK), HORIZON)
    live_train, live_test, live_split = split_train_test(live_ds, TRAIN_FRACTION)
    print('Frontera UTC:', live_split, '| Desarrollo:', len(live_train), '| Validación:', len(live_test))
    display(information_table(live_train, live_test).round(4))
    live_periods = {'desarrollo': live_train, 'validación': live_test}
    live_strategy, live_logs = strategy_table(live_periods, ROUND_TRIP_BPS, Z_THRESHOLD)
    display(live_strategy.round(3))
    display(breakeven_costs(live_logs, live_periods).round(3))
else:
    print('LIVE_E6 = False: no se consulta Kraken. Resultados documentados en la conclusión de abajo.')

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E6 · Kraken BTC/EUR: evidencia insuficiente</b>

**Mercado:** BTC/EUR, Kraken (`XXBTZEUR`), endpoint público OHLC. **Recepción:** 19-11-2026 19:07:14 UTC. **Frecuencia:** 1 minuto. **Cobertura:** 720 velas, de 07:08 a 19:07 UTC (12 horas), **sin huecos**. Precio: 58.139,10 → 57.110,20 € (−177 bps). Autocorrelación de retardo 1: +0,013 (prácticamente nula).

Frontera: 14:56 UTC. Desarrollo: 442 filas; validación: 246 filas.

| Señal | Periodo | IC Pearson | IC Spearman | Alto − bajo |
|---|---|---|---|---|
| Momentum | Desarrollo | −0,093 | −0,099 | −2,47 bps |
| Momentum | Validación | −0,071 | −0,015 | +1,10 bps |
| Reversión | Desarrollo | **+0,045** | +0,077 | +2,70 bps |
| Reversión | Validación | **−0,080** | −0,059 | −10,72 bps |

| Periodo | Regla | Operaciones | Bruto medio | Neto medio (20 bps) | Total neto |
|---|---|---|---|---|---|
| Desarrollo | Momentum | 44 | −1,11 bps | −21,11 bps | −8,9 % |
| Desarrollo | Mean reversion | 20 | +4,68 bps | −15,32 bps | −3,0 % |
| Desarrollo | Periodic long | 74 | +0,15 bps | −19,85 bps | −13,7 % |
| Validación | Momentum | 18 | +1,10 bps | −18,90 bps | −3,4 % |
| Validación | Mean reversion | 14 | +1,40 bps | −18,60 bps | −2,6 % |
| Validación | Periodic long | 41 | −2,50 bps | −22,50 bps | −8,8 % |

- **El IC de la reversión cambia de signo** entre periodos (+0,045 → −0,080), y el de momentum es negativo en los dos.
- Las mejores operaciones brutas (mean reversion, +1,40 bps en validación) quedan **muy lejos** de los 20 bps de coste supuesto.
- Son **12 horas de un solo día**, con 14 a 74 operaciones por caso y objetivos solapados.

**Conclusión: evidencia insuficiente.** No hay base para afirmar que ninguna de las dos señales anticipe el retorno de BTC/EUR, y menos aún que sea rentable después de costes. No se han ajustado parámetros para buscar beneficios.

</div>

### E7 · Hipótesis de investigación: Momentum + Microprice

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E7 · Hipótesis falsable</b>

**Hipótesis (H1).** En BTC/EUR de Kraken, cuando el momentum de 5 minutos es positivo **y** el microprice está por encima del mid (presión compradora en el primer nivel del libro, P02), el retorno del **mid** desde t + 1 s hasta t + 1 s + 60 s es mayor que cuando el momentum es positivo pero el microprice está por debajo del mid.

**H0 (a rechazar):** la diferencia de retorno medio entre los dos grupos es ≤ 0.

**Datos simultáneos que faltan:**
- Snapshots del libro (mejor bid, mejor ask y sus cantidades) con **marca de tiempo del exchange**, a intervalos regulares (por ejemplo cada segundo, por WebSocket), no consultas REST sueltas.
- Trades o velas de 1 minuto **sincronizados por tiempo** con esos snapshots (la vela que cierra en t y el último snapshot recibido antes de t), nunca emparejados por número de fila.
- La hora de recepción local de cada mensaje, para medir la latencia y no usar datos que en realidad llegaron después.
- Varias semanas de captura continua, con los huecos marcados.

**Diseño:**
- **Variables:** momentum de 5 minutos al cierre t; `microprice − mid` en bps con el último snapshot disponible en t.
- **Objetivo:** retorno del mid desde t + 1 s hasta t + 61 s (horizonte de 60 s, con retraso de entrada).
- **Separación:** primeras semanas para desarrollo; las últimas, nunca vistas, para una única validación. Umbrales y horizonte se congelan antes de mirar la validación.

**Criterio de rechazo.** Se rechaza H1 si, en la validación:
- la diferencia de retorno medio entre grupos es ≤ 0, **o**
- su intervalo de confianza al 95 %, calculado con un método que tenga en cuenta el solapamiento (bloques o Newey-West), incluye el 0, **o**
- la diferencia es menor que el coste de ida y vuelta (spread + slippage de P02 + comisiones).

</div>

## 11. Exportación y siguientes pasos
Conservamos indicadores, objetivos y resultados de simulación diferenciados, junto con parámetros y procedencia. Los objetivos son exclusivamente etiquetas. Las funciones `build_features`, `regularize` y `add_target` ya están en `src/quantlab/p05/signals.py`, con sus pruebas en `tests/test_signals.py`. En P06 compararemos Kraken, OKX y Bitvavo.

In [ ]:
if EXPORT:
    folder=Path('quantlab_exports'); folder.mkdir(exist_ok=True)
    prefix=folder/f"p05_{DATA_MODE}_{pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')}"
    dataset.to_csv(str(prefix)+'_signals.csv')
    ic_table.to_csv(str(prefix)+'_ic.csv',index=False)
    strategy_results.to_csv(str(prefix)+'_strategy.csv',index=False)
    cost_table.to_csv(str(prefix)+'_cost.csv',index=False)
    meta={**provenance,'window':WINDOW,'lookback':LOOKBACK,'horizon':HORIZON,'entry_delay_bars':1,'z_threshold':Z_THRESHOLD,'cost_bps':ROUND_TRIP_BPS,'split_time':split_time.isoformat(),'feature_columns':['momentum','sma','ema','rolling_std','price_deviation','z_score'],'target_column':'forward_return','gaps':'no_fill','start':close.index.min().isoformat(),'end':close.index.max().isoformat()}
    Path(str(prefix)+'_metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
    print('Exportado:',prefix)
else: print('Exportación desactivada; activa EXPORT para guardar resultados.')

## 12. Conclusiones

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P05</b>

**¿Qué señal mostró mayor asociación?** Con los parámetros base (N = 20, h = 5), ninguna mostró una asociación positiva sólida. Momentum tuvo el |IC| mayor, pero **negativo** (−0,089 en desarrollo). En la rejilla de desarrollo, la mayor asociación positiva fue la reversión con N = 60 y h = 10 (+0,114).

**¿Persistió en otro periodo?** No. La reversión cambió de signo en validación tanto en la simulación (+0,026 → −0,018) como en Kraken (+0,045 → −0,080). La "mejor" combinación del desarrollo cayó a −0,004 en validación.

**¿Cómo modificaron los costes la interpretación?** Por completo. Los retornos brutos medios están entre −5,5 y +4,7 bps por operación, frente a un coste supuesto de 20 bps. Ninguna regla es rentable neta en ningún periodo, y en la simulación ni siquiera con coste cero.

**¿Qué no se puede concluir con estos datos?**
- Que momentum o mean reversion funcionen, o no funcionen, en BTC/EUR: 12 horas de un día no son evidencia, y la simulación no es Bitcoin.
- La significancia estadística: los objetivos se solapan y no hay intervalos ajustados por dependencia.
- La rentabilidad real: no se modelan spread dinámico, llenados, slippage, impacto ni fiscalidad.
- Que unos parámetros sean "los buenos": se probaron 9 combinaciones y la validación ya se había visto.

**La lección más importante:** la simulación **tiene** dependencia temporal real (autocorrelación de +0,40 y −0,36), y aun así ninguna señal la aprovecha, porque el retraso de una vela en la entrada deja fuera justo el retorno que tenía la información. Una dependencia estadística no es una ventaja explotable.

**No pasar de una señal descriptiva a operar dinero real.**

📄 Resumen en `docs/conclusiones/P05_momentum_mean_reversion.md`

</div>